In [13]:
import pandas as pd
import requests
import re
import os
from io import StringIO

In [14]:
def fetch_and_clean_nasa_csv(url, prefix=""):
    print(f"Extract API NASA untuk: {prefix.strip('_')}")
    response = requests.get(url)
    response.raise_for_status()
    raw_text = response.text
    
    # 1. Pemotongan Statis & Ekstrem
    # Mengabaikan nama kolom, kita memotong data tepat setelah baris "-END HEADER-"
    # Regex \r?\n menangani perbedaan format 'enter' dari server Windows maupun Unix
    data_split = re.split(r"-END HEADER-\r?\n", raw_text)
    
    if len(data_split) < 2:
        raise ValueError("Struktur API tidak dikenali: Tanda '-END HEADER-' tidak ditemukan.")
        
    clean_csv_data = data_split[1]
    
    # 2. Ingesti Data
    df = pd.read_csv(StringIO(clean_csv_data))
    
    # Hapus spasi liar yang sering diselipkan API di awal nama kolom (misal: " T2M")
    df.columns = df.columns.str.strip()
    
    # 3. Dynamic Datetime Feature Engineering
    # Mesin akan beradaptasi dengan output API (MO/DY atau DOY)
    if 'MO' in df.columns and 'DY' in df.columns:
        df['Tanggal'] = pd.to_datetime(df[['YEAR', 'MO', 'DY']].rename(columns={'YEAR': 'year', 'MO': 'month', 'DY': 'day'}))
        df.drop(columns=['YEAR', 'MO', 'DY'], inplace=True)
    elif 'DOY' in df.columns:
        # Konversi: Tahun + Day of Year menjadi format Datetime
        # (%Y%j membaca tahun dan kombinasi hari ke 1 s/d 365)
        df['Tanggal'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j')
        df.drop(columns=['YEAR', 'DOY'], inplace=True)
    else:
        raise KeyError(f"Gagal memparsing waktu. Kolom yang tersedia dari API: {df.columns}")

    # Buang metadata geospasial jika API menyertakannya di dalam tabel
    if 'LAT' in df.columns: df.drop(columns=['LAT'], inplace=True)
    if 'LON' in df.columns: df.drop(columns=['LON'], inplace=True)

    # 4. Finalisasi Format Time-Series
    df.set_index('Tanggal', inplace=True)
    
    if prefix:
        df = df.add_prefix(prefix)
        
    return df

In [15]:
# ==========================================
# EKSEKUSI DATA GARUT & BANDUNG
# ==========================================
url_garut = "https://power.larc.nasa.gov/api/temporal/daily/point?parameters=PRECTOTCORR,T2M,RH2M&community=AG&longitude=107.9022&latitude=-7.2458&start=20220101&end=20251231&format=CSV"
url_bandung = "https://power.larc.nasa.gov/api/temporal/daily/point?parameters=PRECTOTCORR,T2M,RH2M&community=AG&longitude=107.6191&latitude=-6.9175&start=20220101&end=20251231&format=CSV"

# Tarik data Garut
df_nasa_garut = fetch_and_clean_nasa_csv(url_garut, prefix="Garut_")
print("Data Garut diekstrak\n")
print(df_nasa_garut.head())

print("\n" + "="*50 + "\n")

# Tarik data Bandung
df_nasa_bandung = fetch_and_clean_nasa_csv(url_bandung, prefix="Bandung_")
print("Data Bandung diekstrak\n")
print(df_nasa_bandung.head())

Extract API NASA untuk: Garut
Data Garut diekstrak

            Garut_PRECTOTCORR  Garut_T2M  Garut_RH2M
Tanggal                                             
2022-01-01               3.71      23.25       84.15
2022-01-02               7.20      23.11       86.71
2022-01-03               2.65      22.00       88.45
2022-01-04               0.09      22.78       81.91
2022-01-05               4.52      23.36       86.26


Extract API NASA untuk: Bandung
Data Bandung diekstrak

            Bandung_PRECTOTCORR  Bandung_T2M  Bandung_RH2M
Tanggal                                                   
2022-01-01                 0.85        20.99         88.62
2022-01-02                 1.92        21.02         90.47
2022-01-03                 0.56        20.71         87.45
2022-01-04                 0.28        21.33         83.09
2022-01-05                 6.01        21.99         87.82


In [16]:
# ==========================================
# EXPORT KE FILE CSV
# ==========================================

# Simpan data Garut
# index=True agar kolom 'Tanggal' ikut tersimpan
nama_file_garut = "../data/02_interim/NASA_POWER_Garut_Cleaned.csv"
df_nasa_garut.to_csv(nama_file_garut, index=True)
print(f"Berhasil menyimpan: {nama_file_garut}")

# Simpan data Bandung 
nama_file_bandung = "../data/02_interim/NASA_POWER_Bandung_Cleaned.csv"
df_nasa_bandung.to_csv(nama_file_bandung, index=True)
print(f"Berhasil menyimpan: {nama_file_bandung}")

Berhasil menyimpan: ../data/02_interim/NASA_POWER_Garut_Cleaned.csv
Berhasil menyimpan: ../data/02_interim/NASA_POWER_Bandung_Cleaned.csv
